# 06 - Comparaison avec une extraction automatique de variables (tsfresh)

**Question** : des centaines de descripteurs génériques calculés automatiquement (tsfresh) font-ils mieux
que nos variables construites « à la main » avec une justification physique ?

**Comparaison équitable** : mêmes fenêtres (cœur de 1 s), même tâche, mêmes chiens, même protocole
(CV groupée, choix par CV, test à la fin).

**Contraintes de calcul** : tsfresh est lent. On l'applique aux **4 normes** et à un **sous-échantillon du
train** (au plus `MAX_PAR_CLASSE` fenêtres par classe) ; le test est complet. Pour rester équitable,
l'approche 1 est **réentraînée sur le même sous-échantillon**.

| Expérience | Variables | Entraînement |
|---|---|---|
| `approche1_sous_echantillon` | jeu de variables du modèle général retenu | sous-échantillon |
| `tsfresh_1s` | tsfresh sur le cœur de 1 s | sous-échantillon |
| `approche1+tsfresh` | les deux | sous-échantillon |

In [1]:
import os
os.environ["DOG_N_JOBS"] = "2"
import sys
from pathlib import Path

# Racine du projet ajoutée au chemin d'import (le notebook est dans notebooks/)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config as cfg
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import json
from src import data, labels, modeling, evaluation, viz
from src import tsfresh_features as tsf

In [2]:
windows = data.load_table(cfg.WINDOWS_FILE)
dogs = data.load_dogs()
train_ids, test_ids = data.load_split()
pointeur = json.loads((cfg.TABLES_DIR / "meilleur_modele_general.json").read_text())
general = modeling.load_bundle(**pointeur)
classes = list(general["label_encoder"].classes_)

# Même tâche et même filtrage qu'au notebook 04
task = labels.select_task_windows(windows, "general", min_purity=cfg.MIN_PURITY)
task = task.merge(dogs[["DogID"] + cfg.STATIC_FEATURES], on="DogID", how="left")
task = task[task["target"].isin(classes)]
train, test = task[task["DogID"].isin(train_ids)], task[task["DogID"].isin(test_ids)]
print(f"Modèle général : {general['model_name']} / {general['experiment']} | {len(classes)} classes")
print(f"Train : {len(train):,} fenêtres | Test : {len(test):,} fenêtres")

Modèle général : XGBoost / multi_echelle | 11 classes
Train : 93,124 fenêtres | Test : 27,654 fenêtres


In [3]:
MAX_PAR_CLASSE = 1500           # fenêtres de train par classe pour tsfresh
REGLAGES = "efficient"          # "efficient" (~800 variables par signal) ou "minimal" (test rapide)
N_JOBS_TSFRESH = 4
MODELES = ["Dummy_majoritaire", "RandomForest", "XGBoost"]

train_sub = tsf.subsample_per_class(train, MAX_PAR_CLASSE)
print(f"Sous-échantillon de train : {len(train_sub):,} fenêtres")
display(train_sub["target"].value_counts().to_frame("n_fenetres"))

Sous-échantillon de train : 15,095 fenêtres


,n_fenetres
target,
Walking,1500
Panting,1500
Trotting,1500
Lying chest,1500
Playing,1500
Standing,1500
Sniffing,1500
Eating,1500
Sitting,1500


## 1. Extraction tsfresh (mise en cache sur disque)

In [4]:
cache_tr = cfg.PROCESSED_DIR / f"tsfresh_{REGLAGES}_train_{MAX_PAR_CLASSE}.parquet"
cache_te = cfg.PROCESSED_DIR / f"tsfresh_{REGLAGES}_test.parquet"
moves = None
if cache_tr.exists():
    X_tr_raw = data.load_table(cache_tr).set_index("window_id")
else:
    moves = data.load_moves(columns=["DogID", "TestNum", "t_sec"] + cfg.SENSOR_COLUMNS)
    X_tr_raw = tsf.extract_tsfresh(tsf.build_long_format(moves, train_sub), REGLAGES, N_JOBS_TSFRESH)
    data.save_table(X_tr_raw.reset_index(), cache_tr)

if cache_te.exists():
    X_te_raw = data.load_table(cache_te).set_index("window_id")
else:
    if moves is None:
        moves = data.load_moves(columns=["DogID", "TestNum", "t_sec"] + cfg.SENSOR_COLUMNS)
    X_te_raw = tsf.extract_tsfresh(tsf.build_long_format(moves, test), REGLAGES, N_JOBS_TSFRESH)
    data.save_table(X_te_raw.reset_index(), cache_te)
del moves

X_tr, X_te = tsf.impute_from_train(X_tr_raw, X_te_raw)   # imputation apprise sur le train seulement
TSF_COLS = list(X_tr.columns)
print(f"{len(TSF_COLS)} variables tsfresh (après retrait des constantes)")
train_sub = train_sub.merge(X_tr, left_on="window_id", right_index=True, how="inner")
test_t = test.merge(X_te, left_on="window_id", right_index=True, how="inner")

2128 variables tsfresh (après retrait des constantes)


## 2. Entraînement et comparaison (validation croisée)

In [5]:
experiences = {
    "approche1_sous_echantillon": general["features"],
    "tsfresh_1s": TSF_COLS,
    "approche1+tsfresh": general["features"] + TSF_COLS,
}
REPRENDRE = True

resumes, meilleurs = [], {}
for exp, cols in experiences.items():
    b_exp, a_faire = {}, []
    for m in MODELES:
        try:
            if not REPRENDRE:
                raise FileNotFoundError
            b = modeling.load_bundle("general", exp, m)      # échoue si absent ou fichier tronqué
            if b["features"] != list(cols):
                raise ValueError("jeu de variables différent")
            b_exp[m] = b
            print(f"[{exp}] {m} : repris depuis le disque")
        except Exception:
            a_faire.append(m)

    tableau = pd.DataFrame([{
        "experience": exp, "tache": "general", "modele": m,
        "f1_macro_cv_moy": b["cv_mean"], "f1_macro_cv_std": b["cv_std"], "f1_macro_train_moy": np.nan,
        "n_variables": len(cols), "n_variables_retenues": len(modeling.selected_features(b["model"], cols)),
        "meilleurs_params": b["best_params"], "duree_s": np.nan} for m, b in b_exp.items()])
    if a_faire:
        resume, nouveaux = modeling.run_experiment(train_sub, cols, experiment=exp, task="general", models=a_faire)
        b_exp.update(nouveaux)
        tableau = pd.concat([tableau, resume], ignore_index=True)
    resumes.append(tableau.sort_values("f1_macro_cv_moy", ascending=False))
    meilleurs[exp] = modeling.best_bundle(b_exp)

resultats_cv = pd.concat(resumes, ignore_index=True)
resultats_cv.to_csv(cfg.TABLES_DIR / "06_resultats_cv_tsfresh.csv", index=False)
display(resultats_cv.drop(columns=["meilleurs_params"]).round(3))

[approche1_sous_echantillon] Dummy_majoritaire : repris depuis le disque
[approche1_sous_echantillon] RandomForest : repris depuis le disque
[approche1_sous_echantillon] XGBoost : repris depuis le disque
[tsfresh_1s] Dummy_majoritaire : repris depuis le disque
[tsfresh_1s] RandomForest : repris depuis le disque
[tsfresh_1s] XGBoost : repris depuis le disque
[approche1+tsfresh] Dummy_majoritaire : repris depuis le disque
[approche1+tsfresh] RandomForest : repris depuis le disque
[approche1+tsfresh] XGBoost : repris depuis le disque


,experience,tache,modele,f1_macro_cv_moy,f1_macro_cv_std,f1_macro_train_moy,n_variables,n_variables_retenues,duree_s
0,approche1_sous_echantillon,general,XGBoost,0.717,0.051,NaN,282,86,NaN
1,approche1_sous_echantillon,general,RandomForest,0.704,0.049,NaN,282,86,NaN
2,approche1_sous_echantillon,general,Dummy_majoritaire,0.012,0.002,NaN,282,282,NaN
3,tsfresh_1s,general,XGBoost,0.671,0.018,NaN,2128,423,NaN
4,tsfresh_1s,general,RandomForest,0.644,0.025,NaN,2128,423,NaN
5,tsfresh_1s,general,Dummy_majoritaire,0.012,0.002,NaN,2128,2128,NaN
6,approche1+tsfresh,general,XGBoost,0.729,0.040,NaN,2410,456,NaN
7,approche1+tsfresh,general,RandomForest,0.705,0.040,NaN,2410,456,NaN
8,approche1+tsfresh,general,Dummy_majoritaire,0.012,0.002,NaN,2410,2410,NaN


Résultats à remettre en cause par rapport aux écarts type entre plis.... qui dépasse les écarts.

## 3. Évaluation sur le test (mêmes fenêtres pour toutes les expériences)

In [6]:
lignes = []
for exp, b in meilleurs.items():
    pred, _ = modeling.predict(b, test_t)
    lignes.append({"experience": exp, "modele": b["model_name"], "f1_macro_cv": b["cv_mean"],
                   "f1_macro_cv_std": b["cv_std"],
                   **evaluation.compute_metrics(test_t["target"], pred, labels=classes)})
resultats_test = pd.DataFrame(lignes)
resultats_test.to_csv(cfg.TABLES_DIR / "06_resultats_test_tsfresh.csv", index=False)
display(resultats_test.round(3))

b = meilleurs["tsfresh_1s"]
retenues = modeling.selected_features(b["model"], b["features"])
print(f"tsfresh : {len(retenues)} variables retenues par la sélection. Exemples :", retenues[:15])

,experience,modele,f1_macro_cv,f1_macro_cv_std,exactitude,exactitude_equilibree,f1_macro,f1_ponderee,precision_macro,rappel_macro,n_fenetres
0,approche1_sous_echantillon,XGBoost,0.717,0.051,0.786,0.760,0.719,0.794,0.718,0.760,27654
1,tsfresh_1s,XGBoost,0.671,0.018,0.702,0.680,0.638,0.711,0.643,0.680,27654
2,approche1+tsfresh,XGBoost,0.729,0.040,0.796,0.787,0.743,0.805,0.738,0.787,27654


tsfresh : 423 variables retenues par la sélection. Exemples : [np.str_('tsf_GNeck_norm__sum_values'), np.str_('tsf_GNeck_norm__abs_energy'), np.str_('tsf_GNeck_norm__mean_abs_change'), np.str_('tsf_GNeck_norm__median'), np.str_('tsf_GNeck_norm__mean'), np.str_('tsf_GNeck_norm__standard_deviation'), np.str_('tsf_GNeck_norm__variance'), np.str_('tsf_GNeck_norm__root_mean_square'), np.str_('tsf_GNeck_norm__absolute_sum_of_changes'), np.str_('tsf_GNeck_norm__maximum'), np.str_('tsf_GNeck_norm__absolute_maximum'), np.str_('tsf_GNeck_norm__c3__lag_1'), np.str_('tsf_GNeck_norm__c3__lag_2'), np.str_('tsf_GNeck_norm__c3__lag_3'), np.str_('tsf_GNeck_norm__cid_ce__normalize_True')]


**Éléments de discussion** :
- performance : l'extraction générique bat-elle (ou complète-t-elle) les variables expertes ?
- coût : temps d'extraction (minutes à heures) contre quelques secondes pour l'approche 1 ;
- interprétabilité : nos variables ont un sens physique (orientation, fréquence de pas...) ; beaucoup de
  variables tsfresh sont difficiles à interpréter ;
- tsfresh ne voit ici que le cœur de 1 s : l'approche 1 multi-échelle dispose de plus de contexte. C'est un
  choix de coût à mentionner.

In [7]:
from scipy.stats import wilcoxon

scores = {}
for exp, b in meilleurs.items():
    pred, _ = modeling.predict(b, test_t)
    scores[exp] = evaluation.per_dog_scores(test_t.assign(pred=pred)).set_index("DogID")["f1_macro"]
par_chien = pd.DataFrame(scores)
par_chien["gain_tsfresh"] = par_chien["approche1+tsfresh"] - par_chien["approche1_sous_echantillon"]
display(par_chien.round(3))

n_mieux = (par_chien["gain_tsfresh"] > 0).sum()
print(f"Chiens améliorés par l'ajout de tsfresh : {n_mieux} / {len(par_chien)}")
print(f"Gain médian par chien : {par_chien['gain_tsfresh'].median():+.3f}")
print(f"Test de Wilcoxon apparié : p = {wilcoxon(par_chien['gain_tsfresh']).pvalue:.3f}")

C:\Users\Cours APT\AppData\Local\Temp\ipykernel_12840\1419956487.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[exp] = evaluation.per_dog_scores(test_t.assign(pred=pred)).set_index("DogID")["f1_macro"]
C:\Users\Cours APT\AppData\Local\Temp\ipykernel_12840\1419956487.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[exp] = evaluation.per_dog_scores(test_t.assign(pred=pred)).set_index("DogID")["f1_macro"]
C:\Users\Cours APT\AppData\Local\Temp\ipykernel_12840\1419956487.py:6: PerformanceWarning: DataFrame is hi

,approche1_sous_echantillon,tsfresh_1s,approche1+tsfresh,gain_tsfresh
DogID,,,,
16,0.637,0.550,0.694,0.057
30,0.545,0.519,0.623,0.078
33,0.516,0.446,0.553,0.037
34,0.661,0.562,0.687,0.026
41,0.697,0.598,0.701,0.004
55,0.768,0.691,0.793,0.025
70,0.566,0.499,0.636,0.070
73,0.667,0.653,0.668,0.001
74,0.667,0.572,0.690,0.022


Chiens améliorés par l'ajout de tsfresh : 9 / 9
Gain médian par chien : +0.026
Test de Wilcoxon apparié : p = 0.004


### Conclusion : Seul, tsfresh fait moins bien que les variables construites, sur chacun des 9 chiens de test. Combiné à elles, il apporte un gain faible mais systématique (+2,6 points de F1 macro en médiane, 9 chiens sur 9 améliorés, test de Wilcoxon apparié p = 0,004). Ce gain reste petit devant la variabilité entre chiens (environ 25 points), et il coûte environ 1 h 45 de calcul supplémentaire ainsi que de l’interprétabilité. 

### Limites : la comparaison repose sur un seul split, avec 9 chiens. De plus, elle a été faite sur le sous-échantillon de train. Le gain n’est donc pas démontré pour un modèle entraîné sur tout le train